# Preprocessing Pipeline

This notebook prepares Ames Housing data for modeling in a single, readable flow:
1. Setup and data loading
2. Feature grouping and cleaning
3. Skew handling
4. Encoding and transformation
5. Validation checks
6. Test transformation and artifact saving

In [1]:
# 1) Setup: imports
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [2]:
# 2) Load training data and define target
train_path = Path('../train.csv')
if not train_path.exists():
    train_path = Path('train.csv')

df = pd.read_csv(train_path)
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

print(f'Training data loaded from: {train_path.resolve()}')
print(f'X shape: {X.shape} | y shape: {y.shape}')

Training data loaded from: /Users/nirlapidot/Documents/Project new With Copilot/Regression/train.csv
X shape: (1460, 80) | y shape: (1460,)


In [3]:
# 3) Define feature groups based on Ames data description
# Continuous: numeric measured quantities and integer quality/condition scales
continuous_features = [
    'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'GarageArea',
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
    'MiscVal', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MoSold', 'YrSold',
    'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars',
    'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
    'OverallQual', 'OverallCond', 'MSSubClass'
]

# Ordinal: ordered categorical labels
ordinal_features = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC',
    'Fence', 'LotShape', 'LandSlope', 'Functional', 'PavedDrive', 'Street', 'Alley',
    'Utilities', 'CentralAir'
]

# Nominal: unordered categories
nominal_features = [
    'MSZoning', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
    'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation',
    'Heating', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition', 'Id'
]

In [4]:
# 4) Remove high-missing columns and inspect remaining missingness
missing_ratio = X.isnull().mean()
to_drop = missing_ratio[missing_ratio > 0.10].index.tolist()
X = X.drop(columns=to_drop)

print(f"Dropped columns (>10% missing): {to_drop}")

for col in missing_ratio[(missing_ratio > 0) & (missing_ratio <= 0.10)].index:
    col_type = (
        'continuous' if col in continuous_features else
        'ordinal' if col in ordinal_features else
        'nominal' if col in nominal_features else
        'unknown'
    )
    print(f"Column '{col}' has {missing_ratio[col]*100:.1f}% missing values and type: {col_type}")

Dropped columns (>10% missing): ['LotFrontage', 'Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']
Column 'MasVnrArea' has 0.5% missing values and type: continuous
Column 'BsmtQual' has 2.5% missing values and type: ordinal
Column 'BsmtCond' has 2.5% missing values and type: ordinal
Column 'BsmtExposure' has 2.6% missing values and type: ordinal
Column 'BsmtFinType1' has 2.5% missing values and type: ordinal
Column 'BsmtFinType2' has 2.6% missing values and type: ordinal
Column 'Electrical' has 0.1% missing values and type: nominal
Column 'GarageType' has 5.5% missing values and type: nominal
Column 'GarageYrBlt' has 5.5% missing values and type: continuous
Column 'GarageFinish' has 5.5% missing values and type: ordinal
Column 'GarageQual' has 5.5% missing values and type: ordinal
Column 'GarageCond' has 5.5% missing values and type: ordinal


In [5]:
# Ordered category maps used directly by OrdinalEncoder inside the pipeline
ordinal_category_maps = {
    'Street': ['Grvl', 'Pave'],
    'Alley': ['NA', 'Grvl', 'Pave'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NA', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'FireplaceQu': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NA', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NA', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'PavedDrive': ['N', 'P', 'Y'],
    'Utilities': ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    'CentralAir': ['N', 'Y']
}

print(f'Prepared explicit order maps for {len(ordinal_category_maps)} ordinal features.')

Prepared explicit order maps for 23 ordinal features.


In [6]:
# 5) Remove identifier column and keep feature lists aligned
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

if 'Id' in X.columns:
    X = X.drop(columns=['Id'])

if 'Id' in nominal_features:
    nominal_features = [c for c in nominal_features if c != 'Id']

print("'Id' removed from df/X and nominal_features updated.")

'Id' removed from df/X and nominal_features updated.


## 6) Skew Handling Before Preprocessor Fit

We first detect highly skewed continuous features in the training data, then apply `log1p` to safe non-negative columns. This ensures the preprocessor is fit on the transformed training distribution.

In [7]:
# Detect skewed continuous features, transform, then fit preprocessing
skew_threshold = 0.75
continuous_in_X = [c for c in continuous_features if c in X.columns]

feature_skew = X[continuous_in_X].skew(numeric_only=True)
skewed_continuous_features = feature_skew[feature_skew.abs() > skew_threshold].index.tolist()
safe_skewed_features = [c for c in skewed_continuous_features if X[c].min(skipna=True) >= 0]

X_log = X.copy()
for col in safe_skewed_features:
    X_log[col] = np.log1p(X_log[col])

print(f"Skewness threshold: {skew_threshold}")
print(f"Detected skewed continuous features: {skewed_continuous_features}")
print(f"Log-transformed features (non-negative): {safe_skewed_features}")

# Identify nominal subsets after cleaning
binary_nominal_features = [
    col for col in nominal_features if col in X_log.columns and X_log[col].nunique(dropna=True) == 2
]
multi_nominal_features = [
    col for col in nominal_features if col in X_log.columns and X_log[col].nunique(dropna=True) > 2
]

ordinal_in_X = [c for c in ordinal_features if c in X_log.columns]
ordinal_categories = [ordinal_category_maps[c] for c in ordinal_in_X]

continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

binary_nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='if_binary', dtype=int, handle_unknown='ignore'))
])

multi_nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('cont', continuous_pipeline, [f for f in continuous_features if f in X_log.columns]),
    ('ord', ordinal_pipeline, ordinal_in_X),
    ('bin_nom', binary_nominal_pipeline, binary_nominal_features),
    ('multi_nom', multi_nominal_pipeline, multi_nominal_features)
])

X_processed = preprocessor.fit_transform(X_log)
print('Imputation and encoding complete. Shape:', X_processed.shape)
print('Ordinal features with explicit order:', ordinal_in_X)
print('Binary nominal features:', binary_nominal_features)
print('Multi-class nominal features:', multi_nominal_features)

Skewness threshold: 0.75
Detected skewed continuous features: ['LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'KitchenAbvGr', 'BsmtHalfBath', 'MSSubClass']
Log-transformed features (non-negative): ['LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'KitchenAbvGr', 'BsmtHalfBath', 'MSSubClass']
Imputation and encoding complete. Shape: (1460, 206)
Ordinal features with explicit order: ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'GarageFinish', 'GarageQual', 'GarageCond', 'LotShape', 'LandSlope', 'Functional', 'PavedDrive', 'Street', 'Utilities', 'Ce

## 7) Encoding Summary

- Continuous features: mean imputation + standard scaling.
- Ordinal features: most-frequent imputation + `OrdinalEncoder`.
- Binary nominal features: most-frequent imputation + `OneHotEncoder(drop='if_binary')`.
- Multi-class nominal features: most-frequent imputation + `OneHotEncoder(handle_unknown='ignore')`.

In [8]:
# 8) Training transform summary
y_log = np.log1p(y.values)
print(f'X_processed shape: {X_processed.shape}')
print(f'y_log shape: {y_log.shape}')

X_processed shape: (1460, 206)
y_log shape: (1460,)


## 9) Validate Processed Training Matrix

Before transforming test data, verify that the processed training matrix contains no missing values.

In [9]:
# Check for NaNs in X_processed (works for both dense and sparse)
if hasattr(X_processed, 'toarray'):
    nan_count = np.isnan(X_processed.toarray()).sum()
else:
    nan_count = np.isnan(X_processed).sum()

if nan_count == 0:
    print('No missing values remain in X_processed.')
else:
    print(f'Warning: {nan_count} missing values remain in X_processed!')

No missing values remain in X_processed.


## 10) Apply Training-Fit Pipeline to test.csv and Save Artifacts

This section applies the exact train-fitted transformations to `test.csv`, including the same skewed-column log transform, then saves all required artifacts for model training and submission.

In [10]:
# Apply train-fitted preprocessing to test.csv and save final artifacts
# Load test.csv with robust path handling
test_path = Path('../test.csv')
if not test_path.exists():
    test_path = Path('test.csv')

test_df = pd.read_csv(test_path)

# Preserve Id for final submission
id_col = 'Id' if 'Id' in test_df.columns else 'ID'
test_ids = test_df[id_col].copy()

# Build raw test feature matrix and align with training feature space
X_test_raw = test_df.drop(columns=[id_col])
X_test_raw = X_test_raw.drop(columns=[c for c in to_drop if c in X_test_raw.columns], errors='ignore')

train_feature_cols = X_log.columns.tolist()
for col in train_feature_cols:
    if col not in X_test_raw.columns:
        X_test_raw[col] = np.nan
X_test_raw = X_test_raw[train_feature_cols]

# Apply the same skew transform discovered on training data
for col in safe_skewed_features:
    if col in X_test_raw.columns:
        X_test_raw[col] = np.log1p(X_test_raw[col])

# Transform test data with already-fitted preprocessor
X_test_processed = preprocessor.transform(X_test_raw)

# Save all artifacts used by model training notebook
os.makedirs('../data/processed', exist_ok=True)
np.save('../data/processed/X_processed.npy', X_processed)
np.save('../data/processed/X_test_processed.npy', X_test_processed)
np.save('../data/processed/y_log.npy', y_log)
np.save('../data/processed/test_ids.npy', test_ids.to_numpy())

print(f'Training matrix shape: {X_processed.shape}')
print(f'Test matrix shape: {X_test_processed.shape}')
print('Saved artifacts: X_processed.npy, X_test_processed.npy, y_log.npy, test_ids.npy')

Training matrix shape: (1460, 206)
Test matrix shape: (1459, 206)
Saved artifacts: X_processed.npy, X_test_processed.npy, y_log.npy, test_ids.npy
